In [ ]:
"""BigAlpha 2026 v4: proxy-BARRA residual cross-sectional model."""
import os
import numpy as np
import pandas as pd
import dai
from bigalpha_train_v4_submission import MODEL_PATH, predict

def main(datasources, start_date, end_date):
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"missing trained model: {MODEL_PATH}")
    scores = predict(datasources, start_date, end_date, MODEL_PATH)
    universe = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    scores["date"] = pd.to_datetime(scores["date"])
    universe["date"] = pd.to_datetime(universe["date"])
    return (
        pd.merge(scores, universe, on=["date", "instrument"], how="inner")
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"])
        [["date", "instrument", "score"]]
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )

if __name__ == "__main__":
    from bigmodule import M
    datasources = {"bar5m": "bigalpha_2026_stock_bar5m"}
    factor_data = main(datasources, "2024-01-01", "2024-12-31 23:59:59")
    evaluation = M.bigalpha_eval._latest(factor_data=factor_data, show=True)
